# Walmart Trips Hypergraph: scalable centrality and dangling analysis
Vertices are products, hyperedges are shopping trips, and the 11 node labels are product departments.

**Computational design:** full-data sparse hyperdegree, landmark closeness, and incidence-operator eigenvector centrality are computed on all 88,860 products. Exact betweenness and Hypergraph Dangling Centrality are computed on a reproducible 250-product structural core because repeated all-pairs vertex deletion is not computationally defensible on 88,860 vertices.

In [ ]:
import os, zipfile, time, warnings
from itertools import combinations
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix, bmat
from scipy.sparse.csgraph import shortest_path
from scipy.sparse.linalg import LinearOperator, eigsh
sns.set_theme(style='whitegrid', context='notebook')
SEED = 42
rng = np.random.default_rng(SEED)

## 1. Upload and extract `walmart-trips.zip`

In [ ]:
ARCHIVE = '/content/walmart-trips.zip'
if not os.path.exists(ARCHIVE):
    from google.colab import files
    uploaded = files.upload()
    ARCHIVE = next((n for n in uploaded if n.endswith('.zip')), None)
    if ARCHIVE is None: raise ValueError('Please upload walmart-trips.zip')

EXTRACT_DIR = '/content/walmart_data'
os.makedirs(EXTRACT_DIR, exist_ok=True)
with zipfile.ZipFile(ARCHIVE) as zf:
    zf.extractall(EXTRACT_DIR)
DATA_DIR = os.path.join(EXTRACT_DIR, 'walmart-trips')
print(sorted(os.listdir(DATA_DIR)))

## 2. Parse and validate the native hypergraph

In [ ]:
def read_edges(path):
    with open(path, encoding='utf-8') as f:
        return [tuple(map(int, line.strip().split(','))) for line in f if line.strip()]

edges = read_edges(os.path.join(DATA_DIR, 'hyperedges-walmart-trips.txt'))
with open(os.path.join(DATA_DIR, 'node-labels-walmart-trips.txt'), encoding='utf-8') as f:
    node_class = np.array([int(x.strip().split(',')[0]) for x in f if x.strip()])
with open(os.path.join(DATA_DIR, 'label-names-walmart-trips.txt'), encoding='utf-8') as f:
    class_names = [x.strip() for x in f if x.strip()]

nodes = sorted(set().union(*map(set, edges)))
unique_edges = list(dict.fromkeys(tuple(sorted(e)) for e in edges))
edge_sizes = np.array([len(e) for e in edges])
assert len(nodes) == len(node_class) == 88860
assert min(nodes) == 1 and max(nodes) == 88860
assert len(edges) == 69906
print(f'Products: {len(nodes):,}')
print(f'Shopping trips: {len(edges):,}')
print(f'Unique baskets: {len(unique_edges):,}')
print(f'Basket size: min={edge_sizes.min()}, median={np.median(edge_sizes):.0f}, mean={edge_sizes.mean():.2f}, max={edge_sizes.max()}')

## 3. Sparse incidence matrix and full-data hyperdegree

In [ ]:
t0 = time.perf_counter()
rows, cols = [], []
for j, edge in enumerate(edges):
    rows.extend(v-1 for v in edge); cols.extend([j]*len(edge))
H = csr_matrix((np.ones(len(rows), dtype=np.float64), (rows, cols)), shape=(len(nodes), len(edges)))
hyperdegree = np.asarray(H.sum(axis=1)).ravel()
hyperdegree_time = time.perf_counter()-t0
summary = pd.DataFrame({
    'Statistic':['Products','Shopping trips','Unique baskets','Incidences','Mean basket size','Median basket size','Maximum basket size'],
    'Value':[len(nodes),len(edges),len(unique_edges),H.nnz,edge_sizes.mean(),np.median(edge_sizes),edge_sizes.max()]
})
display(summary)
print(f'Incidence construction + hyperdegree: {hyperdegree_time:.3f} s; sparse matrix ≈ {(H.data.nbytes+H.indices.nbytes+H.indptr.nbytes)/1e6:.1f} MB')

## 4. Dataset distributions

In [ ]:
class_count = pd.Series(node_class).value_counts().sort_index()
fig, axes = plt.subplots(1,3,figsize=(18,5))
sns.histplot(edge_sizes, discrete=True, ax=axes[0], color='#E76F51')
axes[0].set(title='Basket-size distribution', xlabel='Products per trip', ylabel='Trips')
sns.histplot(hyperdegree, bins=50, log_scale=(True,True), ax=axes[1], color='#2A9D8F')
axes[1].set(title='Product hyperdegree distribution', xlabel='Trips containing product', ylabel='Products')
axes[2].barh(range(1,12), class_count.values, color=sns.color_palette('viridis',11))
axes[2].set_yticks(range(1,12), class_names); axes[2].invert_yaxis()
axes[2].set(title='Products by department', xlabel='Products')
plt.tight_layout(); plt.savefig('walmart_distributions.png',dpi=300,bbox_inches='tight'); plt.show()

## 5. Full-data eigenvector centrality without forming an 88,860 × 88,860 matrix
The weighted two-section adjacency action is $Ax=H(H^Tx)-D_vx$. A linear operator therefore avoids explicitly storing $A$.

In [ ]:
t0 = time.perf_counter()
def adjacency_matvec(x):
    return H @ (H.T @ x) - hyperdegree*x
Aop = LinearOperator((len(nodes),len(nodes)), matvec=adjacency_matvec, dtype=np.float64)
_, eigvec = eigsh(Aop, k=1, which='LA', tol=1e-6, maxiter=500)
eigenvector = np.abs(eigvec[:,0]); eigenvector /= eigenvector.max()
eigenvector_time = time.perf_counter()-t0
print(f'Full-data eigenvector time: {eigenvector_time:.3f} s')

## 6. Landmark approximation of full-data hypergraph closeness
Distances are computed through the sparse vertex–hyperedge bipartite graph. A vertex-to-vertex hypergraph step equals two bipartite steps. A reachability correction prevents products in small disconnected components from receiving artificially high scores. Set `N_LANDMARKS` higher for a sensitivity analysis.

In [ ]:
N_LANDMARKS = 64
t0 = time.perf_counter()
Zvv = csr_matrix((H.shape[0],H.shape[0])); Zee = csr_matrix((H.shape[1],H.shape[1]))
B = bmat([[Zvv,H],[H.T,Zee]],format='csr')
landmarks = rng.choice(len(nodes),size=N_LANDMARKS,replace=False)
Dbi = shortest_path(B,directed=False,unweighted=True,indices=landmarks)[:,:len(nodes)]
Dh = Dbi/2.0
valid = np.isfinite(Dh) & (Dh>0)
reachable = valid.sum(axis=0)
distance_sum = np.where(valid,Dh,0.0).sum(axis=0)
approx_closeness = (reachable/np.where(distance_sum>0,distance_sum,np.inf)) * (reachable/N_LANDMARKS)
landmark_time = time.perf_counter()-t0
del Dbi, Dh, valid, B
print(f'{N_LANDMARKS}-landmark closeness time: {landmark_time:.3f} s')

## 7. Full-data scalable results
Product identities are anonymized, so interpretation uses product ID and department.

In [ ]:
full_results = pd.DataFrame({
    'product_id':np.arange(1,len(nodes)+1),
    'department_id':node_class,
    'department':[class_names[i-1] for i in node_class],
    'Hyperdegree':hyperdegree,
    'ApproxCloseness64':approx_closeness,
    'Eigenvector':eigenvector
})
display(full_results.nlargest(15,'Hyperdegree'))
display(full_results.nlargest(15,'ApproxCloseness64'))
display(full_results.nlargest(15,'Eigenvector'))

## 8. Reproducible 250-product core for exact comparative validation
The core contains the 250 highest-hyperdegree products. Each original basket is intersected with this core, and intersections of size at least two become core hyperedges. All reported core measures therefore refer specifically to this induced core—not the full Walmart hypergraph.

In [ ]:
CORE_N = 250
core_nodes = (np.argsort(-hyperdegree)[:CORE_N]+1).tolist()
core_set = set(core_nodes)
core_edges = list(dict.fromkeys(tuple(sorted(core_set.intersection(e))) for e in edges if len(core_set.intersection(e))>=2))
Gcore = nx.Graph(); Gcore.add_nodes_from(core_nodes)
for e in core_edges:
    for u,v in combinations(e,2):
        if Gcore.has_edge(u,v): Gcore[u][v]['weight'] += 1
        else: Gcore.add_edge(u,v,weight=1)
print(f'Core: {Gcore.number_of_nodes()} products, {len(core_edges)} hyperedges, {Gcore.number_of_edges()} projected edges, {nx.number_connected_components(Gcore)} components')

## 9. Exact core measures and Hypergraph Dangling Centrality

In [ ]:
def communication_strength_sparse(A):
    D=shortest_path(A,directed=False,unweighted=True)
    upper=D[np.triu_indices(D.shape[0],k=1)]
    valid=np.isfinite(upper) & (upper>0)
    return np.sum(1.0/upper[valid])

timings={'Full hyperdegree':hyperdegree_time,'Full eigenvector':eigenvector_time,'Full landmark closeness':landmark_time}
t=time.perf_counter(); core_degree=dict(Gcore.degree()); timings['Core degree']=time.perf_counter()-t
t=time.perf_counter(); core_close=nx.closeness_centrality(Gcore); timings['Core closeness']=time.perf_counter()-t
t=time.perf_counter(); core_between=nx.betweenness_centrality(Gcore,normalized=True); timings['Core betweenness']=time.perf_counter()-t
t=time.perf_counter(); core_eigen=nx.eigenvector_centrality(Gcore,max_iter=2000,tol=1e-8,weight='weight'); timings['Core eigenvector']=time.perf_counter()-t

Acore=nx.to_scipy_sparse_array(Gcore,nodelist=core_nodes,weight=None,format='csr',dtype=np.float64)
phi0=communication_strength_sparse(Acore); dangling={}
t=time.perf_counter()
for k,removed in enumerate(core_nodes,1):
    Aminus=Acore.tolil(copy=True)
    Aminus[k-1,:]=0; Aminus[:,k-1]=0  # retain removed product as an isolated vertex
    dangling[removed]=(phi0-communication_strength_sparse(Aminus.tocsr()))/phi0
    if k%50==0: print(f'Processed {k}/{CORE_N}')
timings['Core dangling']=time.perf_counter()-t

core_results=pd.DataFrame({'product_id':core_nodes})
core_results['department']=core_results.product_id.map(lambda v:class_names[node_class[v-1]-1])
core_results['Hyperdegree']=core_results.product_id.map(lambda v:hyperdegree[v-1])
core_results['Closeness']=core_results.product_id.map(core_close)
core_results['Betweenness']=core_results.product_id.map(core_between)
core_results['Eigenvector']=core_results.product_id.map(core_eigen)
core_results['Dangling']=core_results.product_id.map(dangling)
display(core_results.nlargest(15,'Dangling'))
display(pd.DataFrame({'Measure':timings.keys(),'Seconds':timings.values()}))

## 10. Readable core visualization and statistical comparison

In [ ]:
top30=core_results.nlargest(30,'Dangling').product_id.tolist()
SG=Gcore.subgraph(top30).copy(); pos=nx.spring_layout(SG,seed=SEED,weight='weight')
plt.figure(figsize=(13,9))
nx.draw_networkx_edges(SG,pos,alpha=.18,edge_color='#457B9D')
vals=[dangling[v] for v in SG]
nx.draw_networkx_nodes(SG,pos,node_size=350,node_color=vals,cmap='magma',edgecolors='white')
for v,(x,y) in pos.items():
    plt.annotate(f'P{v}',(x,y),xytext=(0,7),textcoords='offset points',ha='center',fontsize=7,bbox=dict(facecolor='white',edgecolor='none',alpha=.8,pad=.15))
plt.title('Walmart core: top-30 products by Hypergraph Dangling Centrality')
plt.axis('off'); plt.tight_layout(); plt.savefig('walmart_core_network.png',dpi=300,bbox_inches='tight'); plt.show()

measures=['Hyperdegree','Closeness','Betweenness','Eigenvector','Dangling']
corr=core_results[measures].corr('spearman')
plt.figure(figsize=(7,5.5)); sns.heatmap(corr,annot=True,fmt='.2f',cmap='vlag',center=0,vmin=-1,vmax=1)
plt.title('Walmart core: Spearman rank correlations')
plt.tight_layout(); plt.savefig('walmart_core_correlations.png',dpi=300,bbox_inches='tight'); plt.show()

## 11. Export all results as one ZIP file

In [ ]:
full_results.to_csv('walmart_full_scalable_centralities.csv',index=False)
core_results.to_csv('walmart_core_exact_centralities.csv',index=False)
pd.DataFrame({'Measure':timings.keys(),'Seconds':timings.values()}).to_csv('walmart_runtime.csv',index=False)
summary.to_csv('walmart_dataset_summary.csv',index=False)
outputs=['walmart_full_scalable_centralities.csv','walmart_core_exact_centralities.csv','walmart_runtime.csv','walmart_dataset_summary.csv','walmart_distributions.png','walmart_core_network.png','walmart_core_correlations.png']
with zipfile.ZipFile('Walmart_Hypergraph_results.zip','w',zipfile.ZIP_DEFLATED) as zf:
    for f in outputs: zf.write(f)
print('Saved Walmart_Hypergraph_results.zip')
# In Colab, remove # from the following lines to download:
# from google.colab import files
# files.download('Walmart_Hypergraph_results.zip')

In [ ]:
from google.colab import files
files.download("Walmart_Hypergraph_results.zip")